In [2]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import xarray as xr
import matplotlib.pyplot as plt

# Load ice velocity nc file

**antarctic_ice_vel_phase_map_v01.nc** is 6.9 GB so we only use slices which are 100 MB each.

To convert the VX and VY ice velocity components into magnitude (speed) and direction
(angle), as well as their relative errors, use:

1. speed = sqrt (VX^2 + VY^2)
2. angle = arctan (VY / VX)
3. error = sqrt (ERRX2 + ERRY2)
4. error of flow direction = error / (2*speed)

Caution when inversing the tan().

Based on this a **single-task problem** is to downscale speed, whereas a **multi-task problem** would be to downscale VX and VY. As given above the relationship between (VX, VY) and speed and (VX, VY) and angle is deterministic.

In [93]:
"""
### Load original dataset ###
# Load from whereever it is stored
vel = xr.open_dataset("/Users/kimbente/antarctica_ice_velocity_450m_v2.nc")

### DOMAINS ###
scene_size = 22500 # in meters
n_scenes = 30 # scenes per row/column: 900 scences per domain
span = scene_size * n_scenes

# Transantarctic mountains: Nimrod, Byrd, Skelton glacier, Victoria land
# mountainous domain spanning grounded ice and floating ice.
# stick to order: y, x
transant_y_min = - 1232000
transant_x_min = 3500 # Changed
transant_y_max = transant_y_min + span # -557000
transant_x_max = transant_x_min + span # 678500

# Dome C
# lake Vostok is still further north than this domain
domec_y_min = -1232000
domec_x_min = 899000
domec_y_max = domec_y_min + span # -557000
domec_x_max = domec_x_min + span # 1574000

### Create slices ###
# reduces data size down to managable level
# y slicing ordering is (max, min)
vel_transant_slice = vel.sel(y = slice(transant_y_max, transant_y_min), x = slice(transant_x_min, transant_x_max))
vel_domec_slice = vel.sel(y = slice(domec_y_max, domec_y_min), x = slice(domec_x_min, domec_x_max),)

### Save ###

vel_transant_slice.to_netcdf(path = './nc_data/antarctic_ice_vel_phase_map_v01_TransantarcticMountains_slice.nc')
vel_domec_slice.to_netcdf(path = './nc_data/antarctic_ice_vel_phase_map_v01_DomeC_slice.nc')
"""

In [94]:
vel_transant_slice = xr.open_dataset('./nc_data/antarctic_ice_vel_phase_map_v01_TransantarcticMountains_slice.nc')
vel_domec_slice = xr.open_dataset('./nc_data/antarctic_ice_vel_phase_map_v01_DomeC_slice.nc')

In [98]:
vel_transant_slice

<xarray.Dataset>
Dimensions:       (x: 1501, y: 1501)
Coordinates:
  * x             (x) float64 3.5e+03 3.95e+03 4.4e+03 ... 6.78e+05 6.785e+05
  * y             (y) float64 -5.57e+05 -5.574e+05 ... -1.232e+06 -1.232e+06
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
Attributes: (12/26)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        vel_nsidc.CF16.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    spatial_resolution:        450m
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    license:                   No restrictions on access or use

In [97]:
vel_domec_slice
# One additional x and y value tp define the boundry.

<xarray.Dataset>
Dimensions:       (x: 1501, y: 1501)
Coordinates:
  * x             (x) float64 8.99e+05 8.994e+05 ... 1.574e+06 1.574e+06
  * y             (y) float64 -5.57e+05 -5.574e+05 ... -1.232e+06 -1.232e+06
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
Attributes: (12/26)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        vel_nsidc.CF16.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    spatial_resolution:        450m
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    license:                   No restrictions on access or use

In [5]:
vel_transant_scene = vel_transant_slice.isel(y = slice(0, 50), x = slice(0, 50))

dims = 50

scene_vel_tensor = torch.cat((torch.tensor(vel_transant_scene.VX.values).unsqueeze(0), 
                              torch.tensor(vel_transant_scene.VY.values).unsqueeze(0),
                              torch.tensor(vel_transant_scene.coords["y"].values).unsqueeze(-1).repeat(1, dims).unsqueeze(0),
                              torch.tensor(vel_transant_scene.coords["x"].values).repeat(dims, 1).unsqueeze(0)),
                              dim = 0)

# Save a single scene 50 x 50
torch.save(scene_vel_tensor, './torch_data/scene_vel_tensor.pt')

In [14]:
# speed
torch.sqrt(torch.pow(scene_vel_tensor[0, :, :], exponent = 2) + torch.pow(scene_vel_tensor[1, :, :], exponent = 2)).shape

torch.Size([50, 50])

In [15]:
fig = px.imshow(torch.sqrt(torch.pow(scene_vel_tensor[0, :, :], exponent = 2) + torch.pow(scene_vel_tensor[1, :, :], exponent = 2)), 
                color_continuous_scale = 'RdBu_r',
                origin = "upper", 
                title = "LR bed scene based on upsampling")
fig.show()

In [22]:
domain_dims = 1501

# Domain tensor [C, 1501, 1501]
vel_domec_tensor = torch.cat((torch.tensor(vel_domec_slice.VX.values).unsqueeze(0), 
                              torch.tensor(vel_domec_slice.VY.values).unsqueeze(0),
                              # speed at index 2
                              # torch.sqrt(torch.pow(torch.tensor(vel_domec_slice.VY.values).unsqueeze(0), exponent = 2) + torch.pow(torch.tensor(vel_domec_slice.VX.values).unsqueeze(0), exponent = 2)),
                              # YX
                              torch.tensor(vel_domec_slice.coords["y"].values).unsqueeze(-1).repeat(1, domain_dims).unsqueeze(0),
                              torch.tensor(vel_domec_slice.coords["x"].values).repeat(domain_dims, 1).unsqueeze(0)),
                              dim = 0)

### Conversion to scenes

- IceVelocity scenes are 50 pixels high (H) and wide (W) to achieve the width of 22500.

In [51]:
def domain_to_scenes(domain_tensor, scene_hw = 50):

    n_channels = domain_tensor.shape[0]
    domain_hw = domain_tensor.shape[-1]
    n_hw = int(domain_hw / scene_hw)  

    # Initailise empty scene tensor
    # [N, C, H, W] where n can be split into batches. N (dim 0) can be zero as we concat along this axis.
    scene_tensor = torch.empty(size = (0, n_channels, scene_hw, scene_hw))

    for row in range(0, n_hw):
        row_min = row * scene_hw
        row_max = row_min + scene_hw

        for column in range(0, n_hw):
            column_min = column * scene_hw
            column_max = column_min + scene_hw

            scene_tensor = torch.cat((scene_tensor, domain_tensor[:, row_min : row_max, column_min : column_max].unsqueeze(0)), dim = 0)
    
    return scene_tensor

In [55]:
# Go from [5, 1500, 1500] to [900, 5, 50, 50]: 
# 1500 * 1500 == 900 * 50 * 50
domec_scene_tensor = domain_to_scenes(vel_domec_tensor, scene_hw = 50)
print(domec_scene_tensor.shape)

# third channel is our single-task target
torch.save(scene_vel_tensor, './torch_data/DOMEC_vel_scenes.pt')

In [57]:
# replicate for transant